In [1]:
import pandas as pd
import numpy as np

In [193]:

def load_raw_data(filepath):
    raw_df = pd.read_excel(filepath, header = [0,1])
    return raw_df



In [ ]:
def clean_data_date(raw_df):
    """
    Input : raw DataFrame
    Output: cleaned DataFrame —
            missing values handled, duplicates removed,
            sorted chronologically, consistent schema enforced
    """
    
    #-----------------Date-------------------#
    ##create new column to test check which date cannot be parsed successfully
    raw_df[('real_date','real_date')] = pd.to_datetime(raw_df[('Unnamed: 0_level_0','Dates')], errors = 'coerce')
    raw_df.drop(('Unnamed: 0_level_0','Dates'), axis = 1)
    
    ##-------check any wrong date format--------##
    print(f'sum of NaN in dates = {raw_df[('real_date','real_date')].isna().sum()}')

    ##----------sort datetime----------#
    raw_df = raw_df.sort_values(('real_date', 'real_date'))
    
    ##------drop rows with duplicate date and ticker--------##
    raw_df = raw_df.drop_duplicates(subset = [('real_date', 'real_date')])
    
    ##-------set date as index---------#
    raw_df = raw_df.set_index(('real_date', 'real_date'))
    
    #-------Drop original date---------#
    raw_df = raw_df.drop(("Unnamed: 0_level_0","Dates"), axis = 1)
    
    return raw_df
    
    
    
  

In [195]:
res = clean_data_date(load_raw_data('dummy_data.xlsx')).head()
print(res)


sum of NaN in dates = 0
                       TICK001 US Equity                                   \
                                 PX_LAST PX_VOLUME PX_OPEN PX_LOW PX_HIGH   
(real_date, real_date)                                                      
2020-01-01                           NaN       NaN     NaN    NaN     NaN   
2020-01-02                           NaN       NaN     NaN    NaN     NaN   
2020-01-03                           NaN       NaN     NaN    NaN     NaN   
2020-01-06                           NaN       NaN     NaN    NaN     NaN   
2020-01-07                           NaN       NaN     NaN    NaN     NaN   

                       TICK002 US Equity                                     \
                                 PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW   
(real_date, real_date)                                                        
2020-01-01                    283.053872   915460.0  283.378931  281.846567   
2020-01-02                    289.696332  2

In [196]:
def change_to_long(raw_df):
    return raw_df.stack(level = 0)


res_2 = (change_to_long(res))
print(res_2)

                                             PX_LAST  PX_VOLUME     PX_OPEN  \
(real_date, real_date)                                                        
2020-01-01             TICK001 US Equity         NaN        NaN         NaN   
                       TICK002 US Equity  283.053872   915460.0  283.378931   
                       TICK003 US Equity         NaN        NaN         NaN   
                       TICK004 US Equity   78.154937   773189.0   78.313159   
                       NDX Index          101.483077   395246.0  102.093013   
2020-01-02             TICK001 US Equity         NaN        NaN         NaN   
                       TICK002 US Equity  289.696332  2532517.0  290.124047   
                       TICK003 US Equity         NaN        NaN         NaN   
                       TICK004 US Equity   77.617135  1610706.0   77.355708   
                       NDX Index          103.374469  2543394.0  103.673107   
2020-01-03             TICK001 US Equity         NaN

In [197]:
def drop_non_universal(long_raw_df):
      #------------PRICE--------------#
    #1.check if it is under universal 100 in that year
    long_raw_df.index = long_raw_df.index.set_names(['real_date', 'ticker'])
    
    check_universal = long_raw_df.copy()
    
    ##reset the name of the index
    
    
    
    check_universal = check_universal.reset_index()
    
    ##proportion of non-null/all < 0.5 -> drop
    check_universal = check_universal.drop('real_date', axis = 1).groupby('ticker').apply(lambda x: x.count()/(x.count()+x.isnull().sum())).map(lambda x: x<0.5)
    
    check_universal['to_drop'] = check_universal.any(axis = 1)
    
    to_drop_list = check_universal[check_universal['to_drop']].index.tolist()
    
    
    long_raw_df = long_raw_df[~long_raw_df.index.get_level_values('ticker').isin(to_drop_list)]
    
    
    
    return long_raw_df


    



In [198]:
res_3 = drop_non_universal(res_2)
print(res_3)

                                 PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0 

In [201]:
def check_price_and_volume(long_raw_df):
    
    
    # print(long_raw_df)
    long_raw_df.loc[lambda x: ~((x['PX_LOW'] < x['PX_LAST']) & (x['PX_LAST']  < x['PX_HIGH']) & (x['PX_LOW'] < x['PX_OPEN']) & (x['PX_OPEN']< x['PX_HIGH'])), ['PX_LOW', 'PX_HIGH', 'PX_OPEN', 'PX_LAST']] = None
    
 
    ##---------Track unusual volume with Z-score------------##
    
    mean = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('mean')
    std = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('std')
    long_raw_df['z_score'] = (long_raw_df['PX_VOLUME'] - mean) / std
    long_raw_df.loc[lambda x: x['z_score'].abs() > 3, 'PX_VOLUME'] = None
    long_raw_df = long_raw_df.drop('z_score', axis = 1)
    
    long_raw_df = long_raw_df.ffill(limit = 3)
    
    
    return long_raw_df

res_4 = check_price_and_volume(res_3)

print(res_4)

                                 PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0 

In [ ]:
def check_type(long_raw_df):
    cols = ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME']
    original_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    long_raw_df[cols] = long_raw_df[cols].apply(pd.to_numeric, errors = 'coerce')
    
    after_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    ##-----NaN difference between original and after--------#
    print(original_null - after_null)
    return long_raw_df
    
    
res_5 = check_type(res_4)
print(res_5)


PX_OPEN      0
PX_HIGH      0
PX_LOW       0
PX_LAST      0
PX_VOLUME    0
dtype: int64


PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0   78.103672   78.034933   
           NDX Index          105.178921  2314785.0  105.037270  104.630062   
2020-01-07 TICK002 US Equity  298.194698  2058532.0  299.152835  297.357725   
           TICK004 US Equity   77.869288   862784.0   77.984328   77.782379   
           NDX Index          105.782631  1212384.0  105.804051  105.549470   

                                 PX_HIGH  
real_date  ticker                         
2020-01-01 TICK002 US Equity  283.429203  
           TICK004 US Equity   78.475302  
           NDX Index          102.621097  
2020-01-02 TICK002 US Equity  291.488938  
           TICK004 US Equity   77.686614  
           NDX Index          103.922327  
2020-01-03 TICK002 US Equity  291.797436  
           TICK004 US Equity   79.198394  
           NDX Index          105.457421  
2020-01-06 TICK002 US Equity  297.300806  
           TICK004 US Equity   78.474943  
           NDX Index          105.473471  
2020-01-07 TICK002 US Equity  299.499712  
           TICK004 US Equity   78.772573  
           NDX Index          106.265215

In [ ]:
def check_misalign_date(long_raw_data):
    long_raw_data = long_raw_data.reset_index()

    date_counts = long_raw_data.groupby('real_date')['ticker'].count()
    correct_date =  date_counts[date_counts == date_counts.max()]
    
    correct_date = correct_date.index.tolist()
    
    
    long_raw_data = long_raw_data[long_raw_data['real_date'].isin(correct_date)]
    
    long_raw_data = long_raw_data.set_index(['real_date', 'ticker'])
    
    return long_raw_data

check_misalign_date(res_5)

    

PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0   78.103672   78.034933   
           NDX Index          105.178921  2314785.0  105.037270  104.630062   
2020-01-07 TICK002 US Equity  298.194698  2058532.0  299.152835  297.357725   
           TICK004 US Equity   77.869288   862784.0   77.984328   77.782379   
           NDX Index          105.782631  1212384.0  105.804051  105.549470   

                                 PX_HIGH  
real_date  ticker                         
2020-01-01 TICK002 US Equity  283.429203  
           TICK004 US Equity   78.475302  
           NDX Index          102.621097  
2020-01-02 TICK002 US Equity  291.488938  
           TICK004 US Equity   77.686614  
           NDX Index          103.922327  
2020-01-03 TICK002 US Equity  291.797436  
           TICK004 US Equity   79.198394  
           NDX Index          105.457421  
2020-01-06 TICK002 US Equity  297.300806  
           TICK004 US Equity   78.474943  
           NDX Index          105.473471  
2020-01-07 TICK002 US Equity  299.499712  
           TICK004 US Equity   78.772573  
           NDX Index          106.265215

In [230]:
def add_return_columns(aligned_df):
    """
    Input : aligned price DataFrame
    Output: same DataFrame + simple return and log return columns
            (basic derived series only — no risk/strategy metrics here)
    """
    aligned_df['simple_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform('pct_change')
    
    aligned_df['log_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform(lambda x : np.log(x/x.shift(1)))
    
    aligned_df[['simple_return', 'log_return']] = aligned_df[['simple_return', 'log_return']].fillna(0)
    
    return aligned_df

add_return_columns(check_misalign_date(res_5))

PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0   78.103672   78.034933   
           NDX Index          105.178921  2314785.0  105.037270  104.630062   
2020-01-07 TICK002 US Equity  298.194698  2058532.0  299.152835  297.357725   
           TICK004 US Equity   77.869288   862784.0   77.984328   77.782379   
           NDX Index          105.782631  1212384.0  105.804051  105.549470   

                                 PX_HIGH  simple_return  log_return  
real_date  ticker                                                    
2020-01-01 TICK002 US Equity  283.429203       0.000000    0.000000  
           TICK004 US Equity   78.475302       0.000000    0.000000  
           NDX Index          102.621097       0.000000    0.000000  
2020-01-02 TICK002 US Equity  291.488938       0.023467    0.023196  
           TICK004 US Equity   77.686614      -0.006881   -0.006905  
           NDX Index          103.922327       0.018638    0.018466  
2020-01-03 TICK002 US Equity  291.797436       0.005461    0.005446  
           TICK004 US Equity   79.198394       0.019136    0.018956  
           NDX Index          105.457421       0.018518    0.018348  
2020-01-06 TICK002 US Equity  297.300806       0.014193    0.014094  
           TICK004 US Equity   78.474943      -0.013119   -0.013206  
           NDX Index          105.473471      -0.001043   -0.001043  
2020-01-07 TICK002 US Equity  299.499712       0.009418    0.009374  
           TICK004 US Equity   78.772573      -0.002503   -0.002506  
           NDX Index          106.265215       0.005740    0.005723